# RoBacTutor — QLoRA Fine-Tuning Pipeline
### MSc Computer Science Dissertation — University College Birmingham
**Author:** Vasile Bria | **Student ID:** BRI23222497  
**Supervisor:** Farah Shahid  
**Model:** OpenLLM-Ro/RoMistral-7B-Instruct | **Method:** QLoRA (4-bit NF4)  
**Dataset:** 157 SFT pairs — ANCE Baccalaureate 2024-2025

---
## ⚠️ Before You Start
1. **Runtime → Change runtime type → T4 GPU + High-RAM**
2. **Runtime → Disconnect and delete runtime** (fresh start)
3. Reconnect and run **Cell 1 only**
4. After restart run **Cell 2, 3, 4, 5, 6, 7, 8 in order**
5. Do NOT skip any cell — each one depends on the previous

---
**This is the final, consolidated notebook.** It combines the original dataset/model pipeline and the 5-configuration ablation study (Cell 11, unchanged — these are the numbers already reported in the dissertation's Methodology chapter) with the two proven refinements from later experimentation: early stopping (Cell 7, patience 2, ceiling 10 epochs) and increased LoRA dropout (Cell 6, 0.1) on the **final training run only** (Cell 8). Nothing else was changed from the original.

## Cell 1 — Install
> ⚠️ Runtime restarts after this. Skip this cell after restart.

In [ ]:
# RoBacTutor — Cell 1: Install all packages
# Vasile Bria | BRI23222497 | UCB 2025-2026

!pip install -q \
    "transformers==4.46.0" \
    "bitsandbytes>=0.46.1" \
    "peft==0.12.0" \
    "trl==0.9.6" \
    "accelerate==0.34.2" \
    "datasets==2.21.0" \
    "numpy==1.26.4" \
    "sentencepiece" \
    "rouge-score" \
    "fsspec==2024.6.1"

print('All packages installed!')
print('Restarting runtime...')

import os
os.kill(os.getpid(), 9)


## Cell 2 — Verify Environment
> Must show bnb visible: True before continuing.

In [ ]:
# RoBacTutor — Cell 2: Verify environment
# Vasile Bria | BRI23222497 | UCB 2025-2026

import torch
import transformers
import bitsandbytes as bnb
import numpy as np
from transformers.utils import is_bitsandbytes_available

print('=' * 50)
print('RoBacTutor Fine-Tuning Pipeline')
print('Vasile Bria | BRI23222497 | UCB')
print('=' * 50)
print(f'Python:         {__import__("sys").version.split()[0]}')
print(f'PyTorch:        {torch.__version__}')
print(f'transformers:   {transformers.__version__}')
print(f'bitsandbytes:   {bnb.__version__}')
print(f'numpy:          {np.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'bnb visible:    {is_bitsandbytes_available()}')

if is_bitsandbytes_available() and torch.cuda.is_available():
    print('\n✅ All checks passed — proceed to Cell 3')
else:
    print('\n❌ Something is wrong — re-run Cell 1')


## Cell 3 — Upload Dataset
> Upload `robactutor_sft_dataset_reviewed.jsonl` when prompted.

In [ ]:
# RoBacTutor — Cell 3: Upload dataset
# Vasile Bria | BRI23222497 | UCB 2025-2026

from google.colab import files
import json
from collections import Counter

print('Upload robactutor_sft_dataset_reviewed.jsonl...')
uploaded = files.upload()

DATASET_FILE = list(uploaded.keys())[0]
with open(DATASET_FILE) as f:
    pairs = [json.loads(line) for line in f]

print(f'\nTotal pairs: {len(pairs)}')
for s, c in Counter(p['metadata']['subject'] for p in pairs).items():
    print(f'  {s}: {c}')
print('\n✅ Dataset loaded')


## Cell 4 — Prepare Dataset

In [ ]:
# RoBacTutor — Cell 4: Format and split dataset
# Vasile Bria | BRI23222497 | UCB 2025-2026

from datasets import Dataset
import random

def format_pair(p):
    text = f"<s>[INST] {p['system']}\n\n{p['instruction']} [/INST] {p['response']} </s>"
    return {'text': text}

formatted = [format_pair(p) for p in pairs]
random.seed(42)
random.shuffle(formatted)
split = int(len(formatted) * 0.9)

train_dataset = Dataset.from_list(formatted[:split])
val_dataset   = Dataset.from_list(formatted[split:])

print(f'Train: {len(train_dataset)} examples')
print(f'Val:   {len(val_dataset)} examples')
print('\n✅ Dataset ready')


## Cell 5 — Load RoMistral-7B in 4-bit
> ⏳ First run downloads ~15GB — takes 3-5 minutes.

In [ ]:
# RoBacTutor — Cell 5: Load model in 4-bit QLoRA
# Vasile Bria | BRI23222497 | UCB 2025-2026

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'OpenLLM-Ro/RoMistral-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f'Loading {MODEL_ID} in 4-bit...')
print('Takes 3-5 minutes on first run.')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model loaded!')
print(f'   VRAM used: {used:.1f} / {total:.1f} GB')
print(f'   Device map: {model.hf_device_map}')


## Cell 6 — Apply LoRA Adapters

In [ ]:
# RoBacTutor — Cell 6: Configure LoRA adapters
# Vasile Bria | BRI23222497 | UCB 2025-2026
#
# UPDATED: lora_dropout raised from 0.05 to 0.1. This is a targeted,
# single-variable change in response to overfitting observed in the two
# previous training runs (validation loss best at ~epoch 1, then rising).
# Note this is a genuinely new experiment: the original ablation study
# (Cell 11) only varied rank and learning rate — dropout was fixed at 0.05
# throughout and was never itself tested as a variable until now.

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.1,   # raised from 0.05 — more regularisation, targets the observed overfitting
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied (lora_dropout=0.1)')
print(f'   Trainable params: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'   Frozen params:    {total-trainable:,}')


## Cell 7 — Configure Training

In [ ]:
# RoBacTutor — Cell 7: Training configuration
# Vasile Bria | BRI23222497 | UCB 2025-2026
#
# UPDATED per supervisor feedback ("train it more"): the epoch ceiling is
# raised from 3 to 10, but EarlyStoppingCallback below stops training
# automatically once eval_loss stops improving, rather than blindly running
# all 10. This avoids overfitting the 157-example dataset while still
# giving the model more opportunity to improve than the original 3 epochs.
#
# UPDATED AGAIN: learning_rate changed from 2e-4 to 5e-4. The re-run
# ablation study (Cell 11, with the seed bug fixed) consistently showed
# r=16/lr=5e-4 outperforming r=16/lr=2e-4 (eval loss ~0.48 vs ~0.76 at the
# ablation's 2-epoch budget) across two independent runs. This run tests
# whether that advantage holds over a full early-stopped schedule, not just
# the ablation's short 2-epoch comparison — dropout (0.1) and early
# stopping are otherwise unchanged from the previous best run (0.3268).

from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

training_args = SFTConfig(
    output_dir='./robactutor-qlora',
    num_train_epochs=10,          # raised ceiling (was 3) — early stopping decides the real stopping point
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    fp16=True,
    optim='paged_adamw_8bit',
    learning_rate=5e-4,            # changed from 2e-4 — see note above
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    max_grad_norm=0.3,
    weight_decay=0.001,
    report_to='none',
    seed=42,
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stop after 2 evals with no improvement
)

steps = len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
print(f'✅ Trainer ready!')
print(f'   Learning rate:        {training_args.learning_rate} (changed from 2e-4)')
print(f'   Steps per epoch:      ~{steps}')
print(f'   Epoch ceiling:        {training_args.num_train_epochs} (early stopping patience: 2 evals)')
print(f'   Estimated time:       15-60 minutes, depending on when early stopping triggers')


## Cell 8 — Train! 🚀
> ⏳ Expected: **15-60 minutes** on T4 High-RAM (epoch ceiling raised to 10; early stopping will likely finish sooner).  
> Keep the browser tab open — Colab Pro sessions last 24h.


In [ ]:
# RoBacTutor — Cell 8: Fine-tuning
# Vasile Bria | BRI23222497 | UCB 2025-2026

import time

print('=' * 50)
print('RoBacTutor QLoRA Fine-Tuning')
print('Vasile Bria | BRI23222497 | UCB')
print('=' * 50)
print('Starting training (epoch ceiling 10, early stopping patience 2)...')

start = time.time()
trainer.train()
elapsed = time.time() - start

best_eval = min(h['eval_loss'] for h in trainer.state.log_history if 'eval_loss' in h)
epochs_run = trainer.state.epoch
print(f'\n✅ Training complete!')
print(f'   Time:             {elapsed/60:.1f} minutes')
print(f'   Epochs completed: {epochs_run:.1f} (of {training_args.num_train_epochs} ceiling)')
print(f'   Best eval loss:   {best_eval:.4f}')
if epochs_run < training_args.num_train_epochs:
    print(f'   → Early stopping triggered before reaching the epoch ceiling.')


## Cell 9 — Test: Generate Sample Responses

In [ ]:
# RoBacTutor — Cell 9: Qualitative evaluation
# Vasile Bria | BRI23222497 | UCB 2025-2026

import torch
model.eval()

def generate(instruction, max_new_tokens=300):
    system = (
        'Esti RoBacTutor, un asistent educational specializat in pregatirea elevilor '
        'pentru examenul de Bacalaureat din Republica Moldova.'
    )
    prompt = f'<s>[INST] {system}\n\n{instruction} [/INST]'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

questions = [
    'Subiect: Matematica\n\nCalculati valoarea expresiei: (4/25)^1.5 * (-5)^3.',
    'Subiect: Limba engleza\n\nExplica diferenta dintre present perfect si simple past.',
    'Subiect: Istoria romanilor\n\nExplica termenul domnie in contextul Evului Mediu.',
    "Subiect: Limba și literatura română\n\nContext:\nPROFIL REAL –100 de puncte Citește textul propus și realizează itemii: Primii ani de după război au fost grei... Îmi petreceam timpul liber citind, iar interesul pentru astronomie, fizică și matematică deviase într-o obsesie cronică, așa încât pot zice că mi-am trăit adolescența într-o lume de cifre, legi și formule. Am continuat să învăț la un liceu german, chiar dacă însușisem româna într-a tât de temeinic, încâtdoar cei cu urechea prea fină își dădeau seama că nu sunt român. Eram de departe cel mai b un elev din clasă la toate limbile, însă patima mea erau științele rea le. Vis am să devin \n\nItemul 3: Corectitudinea stilistică L 0 1 2 3 4 5 L 0 1 2 3 4 5",  # Limba romana — real dataset example (was missing before)
]

for i, q in enumerate(questions):
    print(f'\n{"="*55}')
    print(f'Q{i+1}: {q[:70]}')
    print(generate(q))


## Cell 10 — ROUGE Evaluation

In [ ]:
# RoBacTutor — Cell 10: Quantitative evaluation
# Vasile Bria | BRI23222497 | UCB 2025-2026

from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
r1, r2, rL = [], [], []

n_eval = min(20, len(val_dataset))
print(f'Evaluating on {n_eval} validation examples...')

for item in val_dataset.select(range(n_eval)):
    text = item['text']
    inst_end = text.find('[/INST]') + len('[/INST]')
    instruction_part = text[len('<s>[INST] '):text.find('[/INST]')]
    reference = text[inst_end:].replace('</s>', '').strip()
    prediction = generate(instruction_part, max_new_tokens=200)
    s = scorer.score(reference, prediction)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)

print(f'\nRoBacTutor — ROUGE Evaluation')
print(f'Vasile Bria | BRI23222497 | UCB')
print(f'{"-"*40}')
print(f'ROUGE-1: {np.mean(r1):.4f} +/- {np.std(r1):.4f}')
print(f'ROUGE-2: {np.mean(r2):.4f} +/- {np.std(r2):.4f}')
print(f'ROUGE-L: {np.mean(rL):.4f} +/- {np.std(rL):.4f}')


## Cell 11 — Ablation Study
> Change `ABLATION_IDX` (0-4) and re-run for each config. Record results in Cell 12.

In [ ]:
# RoBacTutor — Cell 11: Ablation study (automated — all 5 configs in one run)
# Vasile Bria | BRI23222497 | UCB 2025-2026
#
# FIXED: added set_seed(42) at the top of each loop iteration, BEFORE
# get_peft_model() runs. LoRA's A-matrix is randomly initialised inside
# get_peft_model(), which happens before the Trainer (and its own seed=42)
# is constructed. Without resetting the seed here, each config's
# initialisation silently drifted based on how much randomness the
# previous configs in the loop had already consumed, making the first
# automated run's comparison unfair and irreproducible.

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer
import torch, gc

ABLATION_CONFIGS = [
    {'r': 8,  'lora_alpha': 16, 'lr': 2e-4, 'label': 'r=8,  lr=2e-4'},
    {'r': 16, 'lora_alpha': 32, 'lr': 2e-4, 'label': 'r=16, lr=2e-4 (baseline)'},
    {'r': 32, 'lora_alpha': 64, 'lr': 2e-4, 'label': 'r=32, lr=2e-4'},
    {'r': 16, 'lora_alpha': 32, 'lr': 1e-4, 'label': 'r=16, lr=1e-4'},
    {'r': 16, 'lora_alpha': 32, 'lr': 5e-4, 'label': 'r=16, lr=5e-4'},
]

ablation_results = []

for idx, cfg in enumerate(ABLATION_CONFIGS):
    print('=' * 50)
    print(f'[{idx+1}/5] Running: {cfg["label"]}')
    print('=' * 50)

    # CRITICAL: reset the random seed HERE, before get_peft_model() runs.
    set_seed(42)

    abl_bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16
    )
    abl_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=abl_bnb,
        device_map='auto', trust_remote_code=True
    )
    abl_model = prepare_model_for_kbit_training(abl_model)
    abl_model.enable_input_require_grads()
    abl_model = get_peft_model(abl_model, LoraConfig(
        r=cfg['r'], lora_alpha=cfg['lora_alpha'],
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
    ))

    abl_args = SFTConfig(
        output_dir=f'./ablation_{idx}',
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        fp16=True, optim='paged_adamw_8bit',
        learning_rate=cfg['lr'],
        lr_scheduler_type='cosine', warmup_ratio=0.05,
        logging_steps=5, eval_strategy='epoch',
        save_strategy='no', report_to='none', seed=42,
        dataset_text_field='text', max_seq_length=512, packing=False,
    )

    abl_trainer = SFTTrainer(
        model=abl_model, tokenizer=tokenizer, args=abl_args,
        train_dataset=train_dataset, eval_dataset=val_dataset,
    )
    abl_result = abl_trainer.train()
    abl_eval   = abl_trainer.evaluate()

    print(f'Train loss: {abl_result.training_loss:.4f}')
    print(f'Eval loss:  {abl_eval["eval_loss"]:.4f}\n')

    ablation_results.append({
        'Config': cfg['label'],
        'Train Loss': round(abl_result.training_loss, 4),
        'Eval Loss': round(abl_eval['eval_loss'], 4),
    })

    del abl_trainer, abl_model, abl_bnb
    gc.collect()
    torch.cuda.empty_cache()

print('=' * 50)
print('\u2705 Ablation study complete — all 5 configurations trained.')
print('=' * 50)


## Cell 12 — Ablation Results
> Fill in values after running each config in Cell 11.

In [ ]:
# RoBacTutor — Cell 12: Ablation results table
# Vasile Bria | BRI23222497 | UCB 2025-2026
#
# UPDATED: reads ablation_results (populated automatically by Cell 11)
# instead of manually-typed placeholder values.

import pandas as pd

print('RoBacTutor — Ablation Study Results')
print('Vasile Bria | BRI23222497 | UCB')
print('=' * 50)
df = pd.DataFrame(ablation_results)
print(df.to_string(index=False))

best_idx = df['Eval Loss'].idxmin()
print(f"\n\u2705 Best configuration: {df.loc[best_idx, 'Config']} (eval loss {df.loc[best_idx, 'Eval Loss']:.4f})")


## Cell 13 — Save and Download LoRA Adapters

In [ ]:
# RoBacTutor — Cell 13: Save trained adapters
# Vasile Bria | BRI23222497 | UCB 2025-2026

import os, zipfile
from google.colab import files

SAVE_DIR = './robactutor-lora-adapters'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print('Saved files:')
total_mb = 0
for f in sorted(os.listdir(SAVE_DIR)):
    mb = os.path.getsize(f'{SAVE_DIR}/{f}') / 1e6
    total_mb += mb
    print(f'  {f}: {mb:.1f} MB')
print(f'  Total: {total_mb:.1f} MB')

zip_name = 'robactutor_lora_adapters_BRI23222497.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(SAVE_DIR):
        zf.write(f'{SAVE_DIR}/{fname}', fname)

files.download(zip_name)
print(f'\n✅ Downloaded: {zip_name}')
